# Byzantine Stress Test: Algorithmic Breakdown Point Analysis
### Investigating Robust Aggregation Failure & Agentic Resilience ($f=3$ and $f=4$ Attackers)

This notebook evaluates the **breakdown point** of conventional robust aggregation (`FixedTrimmedMean`) versus our proposed `AgenticAI` under intensified Byzantine adversarial infiltration ($f=3$ and $f=4$ attackers on $N=10$ clients).

### Scientific Motivation & Hypothesis:
1. **Trimmed Mean's Breakdown Point is Exceeded**: With $N=10$ clients and trim ratio $q=0.2$, Trimmed Mean mechanically trims $k = \lfloor 10 \times 0.2 \rfloor = 2$ updates from each coordinate.
   - At **$f=2$ (20% infiltration)**: Both attackers are eliminated $\to$ Trimmed Mean survives (91.82% in previous experiment).
   - At **$f=3$ (30% infiltration)**: Exactly 1 poisoned update leaks into the trimmed mean coordinate $\to$ **catastrophic collapse**.
   - At **$f=4$ (40% infiltration)**: 2 poisoned updates leak into the trimmed mean coordinate $\to$ **catastrophic collapse**.
2. **FixedFedAvg**: Zero Byzantine resilience $\to$ **immediate collapse**.
3. **AgenticAI**: Dynamically flags all 3 or 4 anomalous client profiles, evaluates candidate clean subsets, and selects clean-subset aggregation $\to$ **maintains high accuracy and survives**.

### Quick Setup Instructions:
1. **GPU**: Select `Runtime` $\to$ `Change runtime type` $\to$ `T4 GPU`.
2. **API Key**: Click the Key icon (Secrets) in the left sidebar $\to$ add `GROQ_API_KEY`.
3. Click **Runtime** $\to$ **Run all**.

## 0. Mount Google Drive (Auto-Backup)

In [10]:
from google.colab import drive
import os

drive.mount('/content/drive')
print("Google Drive mounted successfully at /content/drive.")
print("Stress test databases will be backed up to: /content/drive/MyDrive/")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully at /content/drive.
Stress test databases will be backed up to: /content/drive/MyDrive/


In [18]:
import sqlite3, os

db_f3 = '/content/fl_project/fl_metrics_byzantine_stress_f3.db'
if os.path.exists(db_f3):
    con = sqlite3.connect(db_f3)
    rows = con.execute("""
        SELECT runs.method, COUNT(r.id) as rounds_done, MAX(r.round) as latest_round, ROUND(MAX(r.test_accuracy), 2) as max_acc
        FROM experiment3_rounds r
        JOIN experiment3_runs runs ON r.run_id = runs.run_id
        GROUP BY runs.method
    """).fetchall()
    con.close()
    print("=== LIVE EXPERIMENT PROGRESS ===")
    for m, done, latest, acc in rows:
        print(f"  {m:<20}: {done}/20 rounds completed (Latest Round: {latest}, Max Acc: {acc}%)")
else:
    print("Database not found yet. Checking training processes...")
    !ps aux | grep run_experiment3

=== LIVE EXPERIMENT PROGRESS ===
  AgenticAI           : 13/20 rounds completed (Latest Round: 13, Max Acc: 93.98%)
  FixedFedAvg         : 20/20 rounds completed (Latest Round: 20, Max Acc: 94.16%)
  FixedTrimmedMean    : 20/20 rounds completed (Latest Round: 20, Max Acc: 94.26%)


## 1. Verify GPU

In [12]:
!nvidia-smi


Fri Sep 18 17:19:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [13]:
!pip install -q torch torchvision numpy pydantic groq python-dotenv tabulate matplotlib


## 3. Clone Repository or Extract Project

In [14]:
import os, sys, shutil, zipfile

# Priority 1: Check if fl_project_colab.zip was uploaded manually
zip_path = '/content/fl_project_colab.zip'
if os.path.exists(zip_path):
    print("Found uploaded fl_project_colab.zip. Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content')
    print("Extracted project files to /content/fl_project.")
else:
    # Priority 2: Clone fresh from GitHub
    print("No zip found. Cloning repository from GitHub...")
    if os.path.exists('/content/Agentic-AI'):
        shutil.rmtree('/content/Agentic-AI')
    !git clone https://github.com/the-protag0n1st/Agentic-AI.git /content/Agentic-AI

    src_dir = '/content/Agentic-AI/fl_project_final/fl_project'
    dst_dir = '/content/fl_project'
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    print("Project successfully cloned and prepared at /content/fl_project.")

sys.path.insert(0, '/content/fl_project')
os.chdir('/content/fl_project')
print(f"Working directory: {os.getcwd()}")


No zip found. Cloning repository from GitHub...
Cloning into '/content/Agentic-AI'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 131 (delta 33), reused 119 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 678.54 KiB | 8.93 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Project successfully cloned and prepared at /content/fl_project.
Working directory: /content/fl_project


## 4. Configure Groq API Key & Environment

In [15]:
import os, torch
from google.colab import userdata

try:
    groq_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = groq_key
    print("[OK] GROQ_API_KEY successfully loaded from Colab Secrets.")
except Exception:
    if not os.environ.get('GROQ_API_KEY'):
        import getpass
        os.environ['GROQ_API_KEY'] = getpass.getpass("Enter your GROQ_API_KEY: ")

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")


[OK] GROQ_API_KEY successfully loaded from Colab Secrets.
PyTorch Version: 2.11.0+cu128 | CUDA Available: True
Active GPU: Tesla T4


---
## Phase 1 — Stress Test with $f=3$ Attackers (30% Infiltration)
Attacking clients: 3 out of 10 clients inject $-3.0\times$ scaled opposite updates in Rounds 16–20.
- `FixedFedAvg`: Baseline collapse
- `FixedTrimmedMean`: Breakdown point exceeded (only 2 updates trimmed $\to$ 1 poison update leaks)
- `AgenticAI`: Dynamic detection & clean-subset selection

In [16]:
import subprocess, sys, shutil, datetime, os

db_f3 = '/content/fl_project/fl_metrics_byzantine_stress_f3.db'
drive_f3 = '/content/drive/MyDrive/fl_metrics_byzantine_stress_f3.db'

if os.path.exists(drive_f3) and not os.path.exists(db_f3):
    shutil.copy2(drive_f3, db_f3)
    print("Restored f=3 DB from Google Drive.")

def backup_f3():
    try:
        if os.path.exists(db_f3):
            shutil.copy2(db_f3, drive_f3)
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            print(f"[{ts}] DB f=3 successfully backed up to Drive ({os.path.getsize(drive_f3):,} bytes)")
    except Exception as e:
        print(f"Drive backup notice: {e}")

print("=" * 70)
print("RUNNING BASELINES (f=3 Attackers): FixedFedAvg, FixedTrimmedMean")
print("=" * 70)
subprocess.run([
    sys.executable, '-u', 'run_experiment3.py',   # <-- '-u' added here for live streaming output!
    '--seed', '1',
    '--methods', 'FixedFedAvg', 'FixedTrimmedMean',
    '--num-attackers', '3',
    '--rounds', '20',
    '--resume',
    '--db', db_f3,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f3()

RUNNING BASELINES (f=3 Attackers): FixedFedAvg, FixedTrimmedMean
[18:12:55] DB f=3 successfully backed up to Drive (87,842,816 bytes)


In [20]:
print("=" * 70)
print("RUNNING AGENTIC AI (f=3 Attackers): Adaptive Pruning & Aggregation")
print("=" * 70)
subprocess.run([
    sys.executable, 'run_experiment3.py',
    '--seed', '1',
    '--methods', 'AgenticAI',
    '--num-attackers', '3',
    '--rounds', '20',
    '--resume',
    '--db', db_f3,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f3()
print("\n[PHASE 1 COMPLETE] f=3 runs finished!")


RUNNING AGENTIC AI (f=3 Attackers): Adaptive Pruning & Aggregation
[18:39:30] DB f=3 successfully backed up to Drive (132,153,344 bytes)

[PHASE 1 COMPLETE] f=3 runs finished!


In [ ]:
import sqlite3

db_f3 = '/content/fl_project/fl_metrics_byzantine_stress_f3.db'
con = sqlite3.connect(db_f3)
rows = con.execute("""
    SELECT runs.method, r.round, r.test_accuracy, r.aggregation_method, r.selected_clients
    FROM experiment3_rounds r
    JOIN experiment3_runs runs ON r.run_id = runs.run_id
    WHERE r.round IN (15, 16, 20)
    ORDER BY runs.method, r.round
""").fetchall()
con.close()

print(f"{'Method':<20} | {'Round':<6} | {'Test Acc':<10} | {'Aggregation':<12} | Selected Clients")
print("-" * 75)
for m, rnd, acc, agg, clients in rows:
    print(f"{m:<20} | R{rnd:<5} | {acc:>8.2f}% | {agg:<12} | {clients}")

---
## Phase 2 — Stress Test with $f=4$ Attackers (40% Infiltration)
Attacking clients: 4 out of 10 clients inject $-3.0\times$ scaled opposite updates in Rounds 16–20.
- `FixedFedAvg`: Baseline collapse
- `FixedTrimmedMean`: Severe breakdown (only 2 updates trimmed $\to$ 2 poison updates leak)
- `AgenticAI`: Dynamic detection & clean-subset selection of remaining 6 clean clients

In [ ]:
db_f4 = '/content/fl_project/fl_metrics_byzantine_stress_f4.db'
drive_f4 = '/content/drive/MyDrive/fl_metrics_byzantine_stress_f4.db'

if os.path.exists(drive_f4) and not os.path.exists(db_f4):
    shutil.copy2(drive_f4, db_f4)
    print("Restored f=4 DB from Google Drive.")

def backup_f4():
    try:
        if os.path.exists(db_f4):
            shutil.copy2(db_f4, drive_f4)
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            print(f"[{ts}] DB f=4 successfully backed up to Drive ({os.path.getsize(drive_f4):,} bytes)")
    except Exception as e:
        print(f"Drive backup notice: {e}")

print("=" * 70)
print("RUNNING BASELINES (f=4 Attackers): FixedFedAvg, FixedTrimmedMean")
print("=" * 70)
subprocess.run([
    sys.executable, 'run_experiment3.py',
    '--seed', '1',
    '--methods', 'FixedFedAvg', 'FixedTrimmedMean',
    '--num-attackers', '4',
    '--rounds', '20',
    '--resume',
    '--db', db_f4,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f4()


RUNNING BASELINES (f=4 Attackers): FixedFedAvg, FixedTrimmedMean


In [ ]:
print("=" * 70)
print("RUNNING AGENTIC AI (f=4 Attackers): Adaptive Pruning & Aggregation")
print("=" * 70)
subprocess.run([
    sys.executable, 'run_experiment3.py',
    '--seed', '1',
    '--methods', 'AgenticAI',
    '--num-attackers', '4',
    '--rounds', '20',
    '--resume',
    '--db', db_f4,
    '--data-root', '/content/fl_project/data'
], check=True)
backup_f4()
print("\n[PHASE 2 COMPLETE] f=4 runs finished!")


---
## Phase 3 — Results, Comparison Table & Breakdown Point Plot

In [ ]:
import sqlite3
import numpy as np
import matplotlib.pyplot as plt

methods = ['FixedFedAvg', 'FixedTrimmedMean', 'AgenticAI']
display_names = {
    'FixedFedAvg': 'Fixed FedAvg',
    'FixedTrimmedMean': 'Fixed Trimmed Mean (q=0.2)',
    'AgenticAI': 'AgenticAI (Ours)'
}

# 1. Fetch trajectories and Round 20 accuracies
def get_metrics_from_db(db_file):
    if not os.path.exists(db_file):
        return {}
    con = sqlite3.connect(db_file)
    cur = con.cursor()
    data = {}
    for m in methods:
        rows = cur.execute("""
            SELECT r.round, r.test_accuracy
            FROM experiment3_rounds r
            JOIN experiment3_runs runs ON r.run_id = runs.run_id
            WHERE runs.method = ? AND runs.seed = 1
            ORDER BY r.round ASC
        """, (m,)).fetchall()
        if rows:
            data[m] = {r: acc for r, acc in rows}
    con.close()
    return data

data_f3 = get_metrics_from_db('/content/fl_project/fl_metrics_byzantine_stress_f3.db')
data_f4 = get_metrics_from_db('/content/fl_project/fl_metrics_byzantine_stress_f4.db')

# Baseline f=2 results from Experiment 3 Seed 1:
# FixedFedAvg: 87.53%, FixedTrimmedMean: 90.64%, AgenticAI: 91.24%
f2_r20 = {'FixedFedAvg': 87.53, 'FixedTrimmedMean': 90.64, 'AgenticAI': 91.24}

print("=" * 80)
print("BREAKDOWN POINT COMPARISON TABLE (Seed 1, CIFAR-10, N=10)")
print("=" * 80)
print(f"{'Method':<30} | {'f=2 (20% Atk)':>14} | {'f=3 (30% Atk)':>14} | {'f=4 (40% Atk)':>14}")
print("-" * 80)
for m in methods:
    acc_f2 = f2_r20.get(m, 0.0)
    acc_f3 = data_f3.get(m, {}).get(20, None)
    acc_f4 = data_f4.get(m, {}).get(20, None)
    str_f3 = f"{acc_f3:>12.2f}%" if acc_f3 is not None else "     [N/A]  "
    str_f4 = f"{acc_f4:>12.2f}%" if acc_f4 is not None else "     [N/A]  "
    print(f"{display_names[m]:<30} | {acc_f2:>12.2f}% | {str_f3} | {str_f4}")
print("=" * 80)

# 2. Plot Breakdown Curves
f_levels = [2, 3, 4]
colors = {'FixedFedAvg': '#e74c3c', 'FixedTrimmedMean': '#3498db', 'AgenticAI': '#2ecc71'}
markers = {'FixedFedAvg': 'x', 'FixedTrimmedMean': 's', 'AgenticAI': 'o'}

plt.figure(figsize=(10, 5), dpi=120)
for m in methods:
    accs = [
        f2_r20.get(m, np.nan),
        data_f3.get(m, {}).get(20, np.nan),
        data_f4.get(m, {}).get(20, np.nan)
    ]
    plt.plot(f_levels, accs, label=display_names[m], color=colors[m], marker=markers[m], linewidth=2.5, markersize=8)

plt.title("Algorithmic Breakdown Point Comparison (N=10 Clients)", fontsize=14, fontweight='bold')
plt.xlabel("Number of Byzantine Attackers ($f$)", fontsize=12)
plt.ylabel("Final Test Accuracy at Round 20 (%)", fontsize=12)
plt.xticks([2, 3, 4], ["f=2 (20%)", "f=3 (30%)", "f=4 (40%)"])
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=2.0, color='gray', linestyle=':', label="TrimmedMean Breakdown Threshold (q=0.2)")
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("breakdown_point_comparison.png")
plt.show()


---
## 6. Download Stress Test Databases & Plots

In [ ]:
from google.colab import files

for db_name in ['fl_metrics_byzantine_stress_f3.db', 'fl_metrics_byzantine_stress_f4.db']:
    p = f"/content/fl_project/{db_name}"
    if os.path.exists(p):
        print(f"Triggering download for {db_name} ({os.path.getsize(p):,} bytes)...")
        files.download(p)

if os.path.exists("breakdown_point_comparison.png"):
    files.download("breakdown_point_comparison.png")
